# Training Pipeline v7 — Optimised Final Submission
### Combines best findings from Exp1, Exp3, Exp4

| Component | v6 (0.8403) | v7 | Exp score |
|---|---|---|---|
| **Architecture** | 2-layer ConvLSTM · 4-level UNet · CBAM skips · base=32 · dim=64 | **1-layer ConvLSTM · 2-level UNet · no CBAM · base=24 · dim=48** | Exp1-B → 0.8542 |
| **Clustering** | KMeans K=12 | **KMeans K=16** | Exp3-B → 0.8449 |
| **Loss** | SMAPE + 0.3·Corr (ep_weight ramp 1→4) | **α=1.0·GlobalSMAPE + β=3.0·EpSMAPE + γ=0.15·EpCorr** | Exp4-D → 0.8608 |
| **TTA / inference** | E-W flip | E-W flip *(unchanged)* | — |


## 0. Imports & Config

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from sklearn.cluster import KMeans
from scipy.stats import skew as scipy_skew
from statsmodels.tsa.seasonal import STL
import gc, warnings
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
if not torch.cuda.is_available():
    raise RuntimeError('CUDA not available!')
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True
DEVICE = torch.device('cuda:0')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

DATA_ROOT  = Path('/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/raw')
TEST_DIR   = Path('/kaggle/input/competitions/anrf-aise-hack-phase-2-theme-2-pollution-forecasting-iitd/aisehack-theme-2/test_in')
OUTPUT     = Path('/kaggle/working'); OUTPUT.mkdir(exist_ok=True)
LAT_LON    = DATA_ROOT / 'lat_long.npy'
ALL_MONTHS = ['APRIL_16', 'JULY_16', 'OCT_16', 'DEC_16']

LOOKBACK = 10; FORECAST = 16; GRID_H = 140; GRID_W = 124
TRAIN_HOURS = 600

MET_VARS   = ['pblh','u10','v10','rain','t2','q2','psfc','swdown']
EMIS_VARS  = ['SO2','NMVOC_e','PM25','NOx','NH3','NMVOC_finn','bio']
ALL_FEAT   = ['cpm25'] + MET_VARS + EMIS_VARS
LOG_CANDIDATES = {'cpm25','rain','SO2','NMVOC_e','PM25','NOx','NH3','NMVOC_finn','bio','swdown','pblh'}
UNIT_SCALE = {f: 1e9 for f in ['SO2','NMVOC_e','PM25','NOx','NH3','NMVOC_finn','bio']}

# ── v7 CHANGES ───────────────────────────────────────────────────────────────
K_CLUSTERS = 16      # Exp3-B: K=16 (v6 was 12)

# Simplified arch (Exp1-B): no CBAM, 1 LSTM layer, 2 UNet levels
BASE_CHANNELS = 24   # v6 was 32
CONVLSTM_DIM  = 48   # v6 was 64
N_LEVELS      = 2    # v6 was 4
LSTM_LAYERS   = 1    # v6 was 2

# Explicit loss weights (Exp4-D)
ALPHA = 1.0          # GlobalSMAPE weight
BETA  = 3.0          # EpisodeSMAPE weight  (v6 had implicit ramp)
GAMMA = 0.15         # EpisodeCorr weight   (v6 had 0.3)

# ── UNCHANGED FROM V6 ────────────────────────────────────────────────────────
N_CHANNELS  = len(ALL_FEAT) + 4   # 20
GRAD_CLIP   = 1.0
DELTA       = 0.1
TARGET_CH   = 0
BATCH_SIZE  = 16; LR = 3e-4; FIXED_EPOCHS = 28

print(f'K_CLUSTERS={K_CLUSTERS} | base_ch={BASE_CHANNELS} | lstm_dim={CONVLSTM_DIM} | n_levels={N_LEVELS}')
print(f'Loss: α={ALPHA}·GlobalSMAPE + β={BETA}·EpSMAPE + γ={GAMMA}·(1-EpCorr)')
print(f'Channels: {N_CHANNELS} | Batch: {BATCH_SIZE} | Epochs: {FIXED_EPOCHS}')


## 1. Spatial Grid

In [ ]:
ll = np.load(LAT_LON)
lat_grid = ll[..., 0]; lon_grid = ll[..., 1]
igp_mask = ((lat_grid>=24)&(lat_grid<=32)&(lon_grid>=74)&(lon_grid<=88))
print(f'Grid: {lat_grid.shape} | IGP: {igp_mask.sum()} pts ({100*igp_mask.mean():.1f}%)')


## 2. Spatial Clusters — K=16 (Exp3-B)
*(K increased from 12→16; finer spatial granularity)*

In [ ]:
def load_feature(month, feat, root=DATA_ROOT):
    p = root/month/f'{feat}.npy'
    return np.load(p).astype(np.float32) if p.exists() else None

def detect_episodes_grid(arr, period=24, threshold_mult=1.0, subsample=4):
    T,H,W = arr.shape; ep = np.zeros((T,H,W),dtype=bool)
    for ri in range(0,H,subsample):
        for ci in range(0,W,subsample):
            ts = arr[:,ri,ci].astype(float)
            if ts.std()<0.5 or len(ts)<2*period: continue
            try:
                res=STL(ts,period=period,robust=True).fit()
                e=res.resid>res.resid.std()*threshold_mult
                ep[:,ri:min(ri+subsample,H),ci:min(ci+subsample,W)]=e[:,None,None]
            except: continue
    return ep

print('Building cluster features from seasonal PM2.5...')
cl_feats=[]; pm25_mean=np.zeros((GRID_H,GRID_W),np.float32)
pm25_var=np.zeros((GRID_H,GRID_W),np.float32); ep_freq=np.zeros((GRID_H,GRID_W),np.float32)

for month in ALL_MONTHS:
    d = load_feature(month,'cpm25')
    cl_feats.append(np.log1p(d).mean(0).flatten())
    pm25_mean += d.mean(0); pm25_var += d.var(0)
    ep = detect_episodes_grid(d[:TRAIN_HOURS],subsample=4)
    ep_freq += ep.mean(0)
    del d,ep; gc.collect()

pm25_mean/=len(ALL_MONTHS); pm25_var/=len(ALL_MONTHS); ep_freq/=len(ALL_MONTHS)
lat_n=((lat_grid-lat_grid.mean())/lat_grid.std()).flatten()
lon_n=((lon_grid-lon_grid.mean())/lon_grid.std()).flatten()
cl_feats.extend([lat_n,lon_n])
X_cl = np.stack(cl_feats,axis=1)
X_cl = (X_cl-X_cl.mean(0))/(X_cl.std(0)+1e-8)

# K=16 (Exp3-B best)
km = KMeans(n_clusters=K_CLUSTERS, n_init=20, random_state=42)
cluster_labels = km.fit_predict(X_cl)
cluster_map = cluster_labels.reshape(GRID_H,GRID_W).astype(np.int32)
print(f'K-means done ({K_CLUSTERS} clusters).')

def norm01(x): return (x-x.min())/(x.max()-x.min()+1e-8)
means=np.array([pm25_mean[cluster_map==k].mean() for k in range(K_CLUSTERS)])
epfs =np.array([ep_freq[cluster_map==k].mean()   for k in range(K_CLUSTERS)])
stds =np.array([np.sqrt(pm25_var[cluster_map==k].mean()) for k in range(K_CLUSTERS)])
raw_score = 0.4*norm01(means)+0.4*norm01(epfs)+0.2*norm01(stds)
cluster_weights = 0.5 + 2.5*norm01(raw_score)   # same formula as v6

print(f"\n{'Cluster':>8} {'N':>6} {'MeanPM25':>10} {'EpFreq':>8} {'Weight':>8}")
print('-'*46)
for k in np.argsort(-cluster_weights):
    n=int((cluster_map==k).sum())
    print(f"{k:>8} {n:>6} {means[k]:>10.1f} {epfs[k]:>8.3f} {cluster_weights[k]:>8.3f}")

spatial_weight_map = np.zeros((GRID_H,GRID_W),np.float32)
for k in range(K_CLUSTERS): spatial_weight_map[cluster_map==k]=cluster_weights[k]
SPATIAL_W = torch.from_numpy(spatial_weight_map).to(DEVICE)
print(f'\nWeight map: [{spatial_weight_map.min():.3f}, {spatial_weight_map.max():.3f}]')

cluster_map_norm = (cluster_map.astype(np.float32)-K_CLUSTERS/2)/(K_CLUSTERS/2)

fig,axes=plt.subplots(1,3,figsize=(18,5))
axes[0].imshow(cluster_map,origin='lower'); axes[0].set_title(f'Clusters K={K_CLUSTERS}')
axes[1].imshow(spatial_weight_map,origin='lower'); axes[1].set_title('Loss Weights')
axes[2].imshow(pm25_mean,origin='lower'); axes[2].set_title('Mean PM2.5')
plt.tight_layout(); plt.savefig(OUTPUT/'clusters_v7.png',dpi=100); plt.show()


## 3. Normalization

In [ ]:
def compute_norm_stats(all_months, features, train_hours, log_candidates, unit_scale):
    stats={}
    print(f"\n{'Feature':<14} {'RawSkew':>9} {'LogSkew':>9} {'Log?':>5} {'Norm':>12} {'scale':>12}")
    print('-'*66)
    for feat in features:
        vals=[]
        for month in all_months:
            d=load_feature(month,feat)
            if d is None: continue
            chunk=np.maximum(d[:train_hours].flatten(),0)*unit_scale.get(feat,1.0)
            vals.append(chunk)
        if not vals: stats[feat]={'use_log':False,'norm_type':'zscore','loc':0.,'scale':1.}; continue
        raw=np.concatenate(vals); rs=scipy_skew(raw)
        lv=np.log1p(raw); ls=scipy_skew(lv)
        use_log=feat in log_candidates and abs(ls)<abs(rs)*0.8
        work=lv if use_log else raw; std=work.std()
        if std<0.01:
            p1=float(np.percentile(work,1)); p99=float(np.percentile(work,99))
            sc=max(p99-p1,1e-6)
            stats[feat]={'use_log':use_log,'norm_type':'percentile','loc':p1,'scale':sc}
        else:
            loc=float(work.mean()); sc=float(std)+1e-8
            stats[feat]={'use_log':use_log,'norm_type':'zscore','loc':loc,'scale':sc}
        print(f"{feat:<14} {rs:>9.3f} {ls:>9.3f} {'v' if use_log else '':>5} "
              f"{stats[feat]['norm_type']:>12} {stats[feat]['scale']:>12.4f}")
        del raw,vals; gc.collect()
    return stats

print('Computing normalization stats...')
norm_stats = compute_norm_stats(ALL_MONTHS,ALL_FEAT,TRAIN_HOURS,LOG_CANDIDATES,UNIT_SCALE)


## 4. Episode Masks

In [ ]:
print('Computing episode masks...')
ep_masks_full={}
for month in ALL_MONTHS:
    d=load_feature(month,'cpm25'); T=d.shape[0]
    print(f'  {month} T={T}',end=' ... ')
    ep_masks_full[month]=detect_episodes_grid(d)
    print(f'ep={100*ep_masks_full[month].mean():.1f}%')
    del d; gc.collect()

ep_masks_val={}
for month in ALL_MONTHS:
    d=load_feature(month,'cpm25'); T=d.shape[0]; t_s=min(TRAIN_HOURS,T)
    ep_masks_val[month]=detect_episodes_grid(d[t_s:])
    del d; gc.collect()

ep_freq_prior=np.mean([ep_masks_full[m].mean(0) for m in ALL_MONTHS],0)
ep_freq_prior=(ep_freq_prior-ep_freq_prior.mean())/(ep_freq_prior.std()+1e-8)
print(f'Ep-freq prior: [{ep_freq_prior.min():.2f},{ep_freq_prior.max():.2f}]')


## 5. Feature Loading

In [ ]:
def normalize(arr, feat):
    s=norm_stats[feat]
    arr=np.maximum(arr.astype(np.float32),0)*UNIT_SCALE.get(feat,1.0)
    if s['use_log']: arr=np.log1p(arr)
    return (arr-s['loc'])/s['scale']

def build_month_tensor(month):
    channels=[]; ref_T=None
    for feat in ALL_FEAT:
        d=load_feature(month,feat)
        if d is None: d=np.zeros((ref_T,GRID_H,GRID_W),np.float32)
        else: d=normalize(d,feat)
        if ref_T is None: ref_T=d.shape[0]
        channels.append(d[:ref_T])
    T=ref_T; h=np.arange(T)%24
    sin_h=np.broadcast_to(np.sin(2*np.pi*h/24).astype(np.float32)[:,None,None],(T,GRID_H,GRID_W)).copy()
    cos_h=np.broadcast_to(np.cos(2*np.pi*h/24).astype(np.float32)[:,None,None],(T,GRID_H,GRID_W)).copy()
    ep_ch=np.broadcast_to(ep_freq_prior[None],(T,GRID_H,GRID_W)).copy()
    cl_ch=np.broadcast_to(cluster_map_norm[None],(T,GRID_H,GRID_W)).copy()
    channels.extend([sin_h,cos_h,ep_ch,cl_ch])
    tensor=np.stack(channels,axis=1).astype(np.float32)
    am=np.abs(tensor).max()
    print(f'  {month}: {tensor.shape} max_abs={am:.2f}{" ⚠️" if am>1000 else " ✓"}')
    return tensor

print('Building tensors...')
month_tensors={}
for month in ALL_MONTHS: month_tensors[month]=build_month_tensor(month); gc.collect()
assert list(month_tensors.values())[0].shape[1]==N_CHANNELS
print(f'✓ Channels: {N_CHANNELS}')


## 6. Dataset

In [ ]:
class PM25Dataset(Dataset):
    def __init__(self,months,tensors,ep_masks,lb,fc,tc=0,stride=1,aug=False):
        self.lb=lb; self.fc=fc; self.tc=tc; self.aug=aug; self.s=[]
        w=lb+fc
        for m in months:
            t=tensors[m]; e=ep_masks[m]; T=t.shape[0]
            for s in range(0,T-w+1,stride): self.s.append((t,e,s))
        ef=np.mean([ea[s+lb:s+lb+fc].mean() for _,ea,s in self.s])
        print(f'  {len(self.s)} samples | ep={100*ef:.1f}%')
    def __len__(self): return len(self.s)
    def __getitem__(self,i):
        t,e,s=self.s[i]; xe=s+self.lb; ye=xe+self.fc
        x=torch.from_numpy(t[s:xe].copy())
        y=torch.from_numpy(t[xe:ye,self.tc].copy())
        ep=torch.from_numpy(e[xe:ye].astype(np.float32).copy())
        if self.aug and np.random.random()<0.5: x=x.flip(-1); y=y.flip(-1); ep=ep.flip(-1)
        return x,y,ep

class ValDataset(Dataset):
    def __init__(self,months,tensors,ep_masks_v,lb,fc,tc=0,th=600):
        self.lb=lb; self.fc=fc; self.tc=tc; self.s=[]; w=lb+fc
        for m in months:
            t=tensors[m]; T=t.shape[0]; ts=min(th,T)
            tv=t[ts:]; ev=ep_masks_v[m]
            for s in range(0,len(tv)-w+1,2): self.s.append((tv,ev,s))
        print(f'  Val: {len(self.s)} samples')
    def __len__(self): return len(self.s)
    def __getitem__(self,i):
        t,e,s=self.s[i]; xe=s+self.lb; ye=xe+self.fc
        return (torch.from_numpy(t[s:xe].copy()),
                torch.from_numpy(t[xe:ye,self.tc].copy()),
                torch.from_numpy(e[xe:ye].astype(np.float32).copy()))

print('Building datasets:')
train_ds=PM25Dataset(ALL_MONTHS,month_tensors,ep_masks_full,LOOKBACK,FORECAST,TARGET_CH,1,True)
val_ds  =ValDataset(ALL_MONTHS,month_tensors,ep_masks_val,LOOKBACK,FORECAST,TARGET_CH,TRAIN_HOURS)
train_loader=DataLoader(train_ds,batch_size=BATCH_SIZE,shuffle=True,num_workers=4,pin_memory=True,drop_last=True)
val_loader  =DataLoader(val_ds,  batch_size=BATCH_SIZE,shuffle=False,num_workers=2,pin_memory=True)
print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')
xb,yb,ep=next(iter(train_loader))
print(f'x: {xb.shape}')
assert xb.shape[2]==N_CHANNELS


## 7. Model — Simplified Architecture (Exp1-B)

**Changes vs v6:**
- `ConvLSTM`: 2 layers → **1 layer**, hidden dim 64 → **48**
- `UNet`: 4 levels → **2 levels**, base channels 32 → **24**
- **CBAM removed entirely** (skip connections are plain concat)
- Result: ~0.6M params vs v6's ~5M — less overfitting on ~2000 samples


In [ ]:
class ConvBNGELU(nn.Module):
    def __init__(self,ic,oc,k=3,p=1):
        super().__init__()
        self.b=nn.Sequential(nn.Conv2d(ic,oc,k,padding=p,bias=False),nn.BatchNorm2d(oc),nn.GELU(),
                             nn.Conv2d(oc,oc,k,padding=p,bias=False),nn.BatchNorm2d(oc),nn.GELU())
    def forward(self,x): return self.b(x)

class ResBlock(nn.Module):
    def __init__(self,ch):
        super().__init__()
        self.c=nn.Sequential(nn.Conv2d(ch,ch,3,padding=1,bias=False),nn.BatchNorm2d(ch),nn.GELU(),
                             nn.Conv2d(ch,ch,3,padding=1,bias=False),nn.BatchNorm2d(ch))
    def forward(self,x): return F.gelu(x+self.c(x))

class ConvLSTMCell(nn.Module):
    def __init__(self,id_,hd,ks=3):
        super().__init__(); self.hd=hd
        self.conv=nn.Conv2d(id_+hd,4*hd,ks,padding=ks//2,bias=True)
    def forward(self,x,hc):
        h,c=hc; i,f,o,g=self.conv(torch.cat([x,h],1)).chunk(4,1)
        c2=torch.sigmoid(f)*c+torch.sigmoid(i)*torch.tanh(g)
        return torch.sigmoid(o)*torch.tanh(c2),c2
    def init(self,B,H,W,dev): z=torch.zeros(B,self.hd,H,W,device=dev); return z,z

class ConvLSTM(nn.Module):
    def __init__(self,id_,hd,nl=1):
        super().__init__()
        self.cells=nn.ModuleList([ConvLSTMCell(id_ if l==0 else hd,hd) for l in range(nl)])
    def forward(self,seq):
        B,T,C,H,W=seq.shape; hc=[c.init(B,H,W,seq.device) for c in self.cells]
        for t in range(T):
            inp=seq[:,t]
            for l,cell in enumerate(self.cells): h_,c_=cell(inp,hc[l]); hc[l]=(h_,c_); inp=h_
        return hc[-1][0]

class SimplifiedPM25Net(nn.Module):
    """
    v7 architecture (Exp1-B):
      - 1-layer ConvLSTM, hidden dim=48
      - 2-level UNet, base channels=24
      - No CBAM anywhere (plain skip concat)
      - 2 ResBlocks at bottleneck
    """
    def __init__(self,in_ch,base_ch=24,lstm_dim=48,n_levels=2,forecast=16,lstm_layers=1):
        super().__init__(); self.forecast=forecast
        self.temporal=ConvLSTM(in_ch,lstm_dim,nl=lstm_layers)
        self.t_proj=nn.Sequential(nn.Conv2d(lstm_dim,base_ch,1,bias=False),nn.BatchNorm2d(base_ch),nn.GELU())
        self.enc,self.pool,self.enc_ch=nn.ModuleList(),nn.ModuleList(),[]
        ch=base_ch
        for lvl in range(n_levels):
            oc=base_ch*(2**lvl); self.enc.append(ConvBNGELU(ch,oc))
            self.pool.append(nn.MaxPool2d(2)); self.enc_ch.append(oc); ch=oc
        bot=base_ch*(2**n_levels)
        self.bot=nn.Sequential(ConvBNGELU(ch,bot),ResBlock(bot),ResBlock(bot))
        self.up,self.dec=nn.ModuleList(),nn.ModuleList()
        ch=bot
        for lvl in reversed(range(n_levels)):
            sc=self.enc_ch[lvl]; self.up.append(nn.ConvTranspose2d(ch,sc,2,2))
            self.dec.append(ConvBNGELU(sc*2,sc)); ch=sc
        self.head=nn.Sequential(nn.Conv2d(ch,ch,3,padding=1,bias=False),nn.BatchNorm2d(ch),nn.GELU(),nn.Conv2d(ch,forecast,1))
        for m in self.modules():
            if isinstance(m,nn.Conv2d): nn.init.kaiming_normal_(m.weight,nonlinearity='relu'); (nn.init.zeros_(m.bias) if m.bias is not None else None)
            elif isinstance(m,nn.BatchNorm2d): nn.init.ones_(m.weight); nn.init.zeros_(m.bias)

    def forward(self,x):
        pers=x[:,-1,0:1]
        h=self.temporal(x); feat=self.t_proj(h); skips=[]
        for enc,pool in zip(self.enc,self.pool): feat=enc(feat); skips.append(feat); feat=pool(feat)
        feat=self.bot(feat)
        for up,dec,skip in zip(self.up,self.dec,reversed(skips)):
            feat=up(feat)
            if feat.shape[-2:]!=skip.shape[-2:]: feat=F.interpolate(feat,skip.shape[-2:],mode='bilinear',align_corners=False)
            feat=dec(torch.cat([feat,skip],1))
        return pers+self.head(feat)

model=SimplifiedPM25Net(N_CHANNELS,BASE_CHANNELS,CONVLSTM_DIM,N_LEVELS,FORECAST,LSTM_LAYERS).to(DEVICE)
np_=sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'v7 Parameters: {np_:,} ({np_/1e6:.2f}M)  [v6 was ~5M]')
with torch.no_grad():
    d=torch.randn(2,LOOKBACK,N_CHANNELS,GRID_H,GRID_W,device=DEVICE)
    o=model(d); assert o.shape==(2,FORECAST,GRID_H,GRID_W)
    print(f'Forward: {d.shape} → {o.shape} ✓')
del d,o


## 8. Loss & Metrics — Explicit α/β/γ Weights (Exp4-D)

**Changes vs v6:**
- Loss explicitly separated into 3 terms: `α·GlobalSMAPE + β·EpSMAPE + γ·(1-EpCorr)`
- `β=3.0` (heavy EpisodeSMAPE penalty) vs v6's implicit episode ramp
- `γ=0.15` (lighter correlation term) vs v6's `0.3`
- Episode ramp removed — β is fixed throughout training


In [ ]:
def competition_loss(pred, target, ep_mask, spatial_w,
                    alpha=ALPHA, beta=BETA, gamma=GAMMA,
                    delta=DELTA, eps=1e-8):
    """
    v7 loss: α·GlobalSMAPE(spatial_w) + β·EpisodeSMAPE + γ·(1-EpisodeCorr)
    GlobalSMAPE : smooth SMAPE over ALL points, weighted by spatial_w
    EpisodeSMAPE: smooth SMAPE computed ONLY over episodic points
    EpisodeCorr : Pearson correlation over episodic points
    """
    diff = pred - target
    smooth_abs = torch.sqrt(diff**2 + delta**2) - delta
    denom = 0.5*(target.abs()+pred.abs()) + eps
    smape_elem = smooth_abs / denom   # (B, T, H, W)

    # Global SMAPE — all points, spatial cluster weights
    w_global = spatial_w[None, None]  # (1,1,H,W)
    L_global = (w_global * smape_elem).sum() / w_global.sum() / pred.shape[0] / pred.shape[1]

    # Episode SMAPE & Pearson — episodic points only
    ep_bool = ep_mask.bool()
    n_ep = ep_bool.sum()
    if n_ep > 10:
        L_episode = smape_elem[ep_bool].mean()
        p_ep = pred[ep_bool]; t_ep = target[ep_bool]
        pm = p_ep.mean(); tm = t_ep.mean()
        cov = ((p_ep-pm)*(t_ep-tm)).mean()
        ps = (p_ep-pm).pow(2).mean().sqrt().clamp(min=eps)
        ts_ = (t_ep-tm).pow(2).mean().sqrt().clamp(min=eps)
        L_corr = 1.0 - cov/(ps*ts_)
    else:
        L_episode = torch.tensor(0., device=pred.device)
        L_corr    = torch.tensor(0., device=pred.device)

    total = alpha*L_global + beta*L_episode + gamma*L_corr
    return total, L_global.item(), L_episode.item(), L_corr.item()


def denorm_pm25(arr):
    s=norm_stats['cpm25']; out=arr*s['scale']+s['loc']
    if s['use_log']: out=np.expm1(np.clip(out,-10,20))
    return np.maximum(out,0)

@torch.no_grad()
def evaluate_competition(model, loader):
    model.eval(); g_list,e_list,c_list=[],[],[]
    for xb,yb,ep in loader:
        p=denorm_pm25(model(xb.to(DEVICE)).cpu().numpy())
        t=denorm_pm25(yb.numpy()); ep_np=ep.numpy().astype(bool); B,T_,H,W=p.shape
        dg=0.5*(np.abs(p)+np.abs(t)); g_list.append((np.abs(p-t)/(dg+1e-8)).mean())
        for b in range(B):
            for ti in range(T_):
                mask=ep_np[b,ti]
                if mask.sum()<2: continue
                pe=p[b,ti][mask]; te=t[b,ti][mask]
                de=0.5*(np.abs(pe)+np.abs(te)); e_list.append((np.abs(pe-te)/(de+1e-8)).mean())
                if len(pe)>5 and pe.std()>1e-8 and te.std()>1e-8:
                    r=np.corrcoef(pe,te)[0,1]
                    if not np.isnan(r): c_list.append(r)
    model.train()
    g=float(np.mean(g_list)); e=float(np.mean(e_list)) if e_list else float('nan')
    c=float(np.mean(c_list)) if c_list else float('nan')
    return {'g':g,'e':e,'c':c,'ng':1-g/2,
            'ne':1-e/2 if not np.isnan(e) else float('nan'),
            'nc':(c+1)/2 if not np.isnan(c) else float('nan'),
            'g%':g*100,'e%':e*100}

print(f'v7 loss: α={ALPHA}·GlobalSMAPE + β={BETA}·EpSMAPE + γ={GAMMA}·(1-EpCorr)')
print('Metrics defined.')


## 9. Training

In [ ]:
def train_epoch(model,loader,opt,sched):
    model.train(); total=0.; lg_sum=0.; le_sum=0.; lc_sum=0.
    for xb,yb,ep in loader:
        xb=xb.to(DEVICE,non_blocking=True); yb=yb.to(DEVICE,non_blocking=True); ep=ep.to(DEVICE,non_blocking=True)
        opt.zero_grad(set_to_none=True)
        loss,lg,le,lc=competition_loss(model(xb),yb,ep,SPATIAL_W)
        loss.backward(); nn.utils.clip_grad_norm_(model.parameters(),GRAD_CLIP)
        opt.step()
        if sched: sched.step()
        total+=loss.item(); lg_sum+=lg; le_sum+=le; lc_sum+=lc
    n=len(loader)
    return total/n, lg_sum/n, le_sum/n, lc_sum/n

opt=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=1e-4)
sch=torch.optim.lr_scheduler.OneCycleLR(opt,max_lr=LR,steps_per_epoch=len(train_loader),
                                         epochs=FIXED_EPOCHS,pct_start=0.1)
history={'loss':[],'ng':[],'ne':[],'nc':[]}

print('='*80)
print(f'v7 Training: {FIXED_EPOCHS} epochs | {np_/1e6:.2f}M params | K={K_CLUSTERS} clusters')
print(f'Loss: α={ALPHA}·GlobalSMAPE + β={BETA}·EpSMAPE + γ={GAMMA}·(1-EpCorr)  [no ep ramp]')
print('Val metrics: ng/ne/nc in [0,1] higher=better')
print('='*80)

for epoch in range(FIXED_EPOCHS):
    tl,lg,le,lc=train_epoch(model,train_loader,opt,sch)
    m=evaluate_competition(model,val_loader)
    history['loss'].append(tl); history['ng'].append(m['ng'])
    history['ne'].append(m['ne']); history['nc'].append(m['nc'])
    print(f"Ep {epoch+1:03d}/{FIXED_EPOCHS} | "
          f"L={tl:.4f}(g={lg:.3f} ep={le:.3f} cr={lc:.3f}) | "
          f"ng={m['ng']:.4f} ne={m['ne']:.4f} nc={m['nc']:.4f} | "
          f"g%={m['g%']:.1f} e%={m['e%']:.1f} c={m['c']:.4f}")
    if (epoch+1)%5==0:
        torch.save(model.state_dict(),OUTPUT/f'v7_ep{epoch+1:03d}.pt')
        print('  → ckpt saved')

torch.save(model.state_dict(),OUTPUT/'v7_final.pt')
best_ng=max(history['ng'])
best_ne=max(v for v in history['ne'] if not np.isnan(v))
best_nc=max(v for v in history['nc'] if not np.isnan(v))
print(f'\n✓ Done. Best → ng={best_ng:.4f} ne={best_ne:.4f} nc={best_nc:.4f}')
print(f'  v6 baseline: 0.8403 | Exp1-B: 0.8542 | Exp3-B: 0.8449 | Exp4-D: 0.8608')

# Plot training curves
fig,axes=plt.subplots(1,3,figsize=(15,4))
for ax,key,label in zip(axes,['ng','ne','nc'],['NormGlobalSMAPE','NormEpSMAPE','NormEpCorr']):
    vals=[v for v in history[key] if not (isinstance(v,float) and v!=v)]
    ax.plot(vals,label='v7'); ax.axhline(0.8403,color='r',linestyle='--',label='v6 LB')
    ax.set_title(label); ax.set_xlabel('Epoch'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(OUTPUT/'training_curves_v7.png',dpi=100); plt.show()


## 10. Inference with TTA + Save

In [ ]:
def load_test_tensor(test_dir):
    channels=[]; ref_shape=None
    for feat in ALL_FEAT:
        p=test_dir/f'{feat}.npy'
        arr=np.load(p).astype(np.float32) if p.exists() else np.zeros(ref_shape,dtype=np.float32)
        if ref_shape is None: ref_shape=arr.shape
        s=norm_stats[feat]
        arr=np.maximum(arr,0)*UNIT_SCALE.get(feat,1.0)
        if s['use_log']: arr=np.log1p(arr)
        arr=(arr-s['loc'])/s['scale']
        channels.append(arr)
    N,T,H,W=ref_shape
    sin_ch=np.zeros((N,T,H,W),np.float32); cos_ch=np.ones((N,T,H,W),np.float32)
    tp=test_dir/'time.npy'
    if tp.exists():
        try:
            times=np.load(tp); hh=times%24
            sin_ch=(np.sin(2*np.pi*hh/24)[:,:,None,None]*np.ones((N,T,H,W))).astype(np.float32)
            cos_ch=(np.cos(2*np.pi*hh/24)[:,:,None,None]*np.ones((N,T,H,W))).astype(np.float32)
            print('  time.npy hour encoding ✓')
        except: pass
    ep_ch=np.broadcast_to(ep_freq_prior[None,None],(N,T,H,W)).copy().astype(np.float32)
    cl_ch=np.broadcast_to(cluster_map_norm[None,None],(N,T,H,W)).copy().astype(np.float32)
    channels.extend([sin_ch,cos_ch,ep_ch,cl_ch])
    tensor=np.stack(channels,axis=2).astype(np.float32)
    am=np.abs(tensor).max()
    print(f'Test tensor: {tensor.shape} max_abs={am:.2f}{" ⚠️" if am>1000 else " ✓"}')
    return tensor

print('Loading test data...')
test_tensor=load_test_tensor(TEST_DIR)
N_test=test_tensor.shape[0]
assert test_tensor.shape[2]==N_CHANNELS

# TTA: original + E-W flip
model.eval(); p_orig=[]; p_flip=[]
with torch.no_grad():
    for s in range(0,N_test,8):
        e=min(s+8,N_test)
        b=torch.from_numpy(test_tensor[s:e]).float().to(DEVICE)
        p_orig.append(denorm_pm25(model(b).cpu().numpy()))
        p_flip.append(denorm_pm25(model(b.flip(-1)).cpu().numpy())[...,::-1].copy())
        print(f'  {e}/{N_test}',end='\r')

preds=0.5*np.concatenate(p_orig,0)+0.5*np.concatenate(p_flip,0)
print(f'\nTTA predictions: {preds.shape}')

train_max=max(float(load_feature(m,'cpm25').max()) for m in ALL_MONTHS)
preds=np.clip(preds,0,train_max*1.25)
preds=np.nan_to_num(preds,nan=0.)

preds_submit=preds.transpose(0,2,3,1)
assert preds_submit.shape==(218,140,124,16)
np.save(OUTPUT/'preds.npy',preds_submit)
assert np.load(OUTPUT/'preds.npy').shape==(218,140,124,16)
print(f'Saved preds.npy | [{preds_submit.min():.1f}, {preds_submit.max():.1f}] µg/m³')
print('✓ Ready to submit')